# Unsloth Gemma4 — MIMIC-III Multitask Fine-Tuning Pipeline

**Control panel notebook for OSC OnDemand Jupyter.**

This notebook is a SLURM-driven control panel. Training, inference, and scoring all run on GPU compute nodes via `sbatch`.
The notebook itself only:
- validates the dataset and environment,
- displays audit reports,
- monitors job status,
- loads and interprets results,
- and gates full training behind a strict parse coverage check.

**GPU is not visible on the login/OnDemand node.** `unsloth` cannot be imported here. All heavy computation is submitted to SLURM.

---

## Pipeline overview

```
Section 0  Config & helpers
Section 1  Environment check
Section 2  Dataset validation
Section 3  Tokenization & label-mask audit reports
Section 4  Current SLURM job status
Section 5  Load smoke results
Section 6  Strict parse coverage gate
Section 7  [CONDITIONAL] Strict-v2 dataset branch (if gate fails)
Section 8  [SLURM] Submit smoke training
Section 9  [SLURM] Submit smoke inference + scoring
Section 10 [SLURM] Full training (gate must pass first)
Section 11 Experiment summary
```

Cells marked **[SLURM]** require the user to un-comment a line before they submit a job.

---
## Section 0 — Imports, Config, and Helpers

In [14]:
import importlib.metadata
import json
import os
import subprocess
import sys
import textwrap
from datetime import datetime
from pathlib import Path

import pandas as pd

# ---------------------------------------------------------------------------
# Project constants
# ---------------------------------------------------------------------------
PROJECT_DIR          = Path("/users/PCS0229/imankhazrak/EHR-Agentic-AI")
CONDA_ENV            = "unsloth_env"
BASE_MODEL           = "unsloth/gemma-4-e2b-it-unsloth-bnb-4bit"
TRAIN_JSONL          = PROJECT_DIR / "dataset_for_unsloth/train.jsonl"
TEST_JSONL           = PROJECT_DIR / "dataset_for_unsloth/test.jsonl"

# Smoke run parameters
SMOKE_TRAIN_SAMPLES  = 500
SMOKE_EVAL_SAMPLES   = 50
SMOKE_MAX_STEPS      = 500
SMOKE_TEST_SAMPLES   = 50

# Gate: minimum strict parse coverage required before full training
STRICT_PARSE_GATE    = 0.80

# Full training parameters
FULL_MAX_STEPS       = 1500
FULL_SAVE_STEPS      = 100

# SLURM account / partition
ACCOUNT              = "pcs0229"
PARTITION            = "gpu-exp"

# Known 500-step smoke run directory
# *** UPDATE THIS after every new smoke training run ***
# Set to the new timestamped directory printed at the end of the train job log (OUT_DIR=...).
SMOKE_RUN_DIR = PROJECT_DIR / "outputs/unsloth_mt_lora_response_only_smoke_20260520_185354"

# SLURM job ID for the inference+scoring job associated with SMOKE_RUN_DIR above
# *** UPDATE THIS whenever you submit a new test_unsloth_response_only_smoke.slurm job ***
SMOKE_TEST_JOB_ID    = "47576605"

# Change working directory so all relative paths resolve correctly
os.chdir(PROJECT_DIR)
print(f"Working directory: {Path.cwd()}")


# ---------------------------------------------------------------------------
# Helper functions
# ---------------------------------------------------------------------------

def run(cmd: str, capture: bool = True, check: bool = False) -> str:
    """Run a shell command and return stdout+stderr as a string."""
    result = subprocess.run(
        cmd, shell=True, capture_output=capture,
        text=True, cwd=str(PROJECT_DIR)
    )
    output = (result.stdout or "") + (result.stderr or "")
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed (rc={result.returncode}):\n{cmd}\n{output}")
    return output.strip()


def check_file(path, label: str = None) -> bool:
    """Print whether path exists and return the boolean result."""
    p = Path(path)
    exists = p.exists()
    tag = label or str(p.relative_to(PROJECT_DIR) if p.is_absolute() and p.is_relative_to(PROJECT_DIR) else p)
    status = "OK" if exists else "MISSING"
    print(f"  [{status}] {tag}")
    return exists


def load_jsonl(path) -> list:
    """Load a JSONL file into a list of dicts."""
    p = Path(path)
    if not p.is_file():
        raise FileNotFoundError(f"JSONL not found: {p}")
    return [json.loads(line) for line in p.read_text().splitlines() if line.strip()]


def load_json(path) -> dict:
    """Load a JSON file into a dict."""
    p = Path(path)
    if not p.is_file():
        raise FileNotFoundError(f"JSON not found: {p}")
    return json.loads(p.read_text())


def tail_file(path, n: int = 80) -> str:
    """Return the last n lines of a text file."""
    p = Path(path)
    if not p.is_file():
        return f"[File not found: {p}]"
    lines = p.read_text().splitlines()
    return "\n".join(lines[-n:])


def show_predictions(rows: list, n: int = 5, label: str = "Predictions") -> None:
    """Print the first n prediction_text values from a predictions JSONL."""
    print(f"\n--- {label} (first {n}) ---")
    for r in rows[:n]:
        idx = r.get("index", "?"); pair = r.get("pair_id", "?")
        ok  = r.get("pred_parse_ok", None)
        txt = str(r.get("prediction_text") or "")[:400]
        print(f"  index={idx}  pair_id={pair}  pred_parse_ok={ok}")
        print(textwrap.indent(txt, "    "))
        print()


def show_failed_predictions(rows: list, n: int = 5) -> None:
    """Print the first n rows where pred_parse_ok is False."""
    failed = [r for r in rows if not r.get("pred_parse_ok", True)]
    print(f"\n--- Failed predictions: {len(failed)} total (showing first {n}) ---")
    if not failed:
        print("  No failed predictions.")
        return
    for r in failed[:n]:
        idx = r.get("index", "?"); pair = r.get("pair_id", "?")
        txt = str(r.get("prediction_text") or "")[:500]
        print(f"  index={idx}  pair_id={pair}")
        print(textwrap.indent(txt, "    "))
        print()


print("Config loaded.")
print(f"  BASE_MODEL       = {BASE_MODEL}")
print(f"  TRAIN_JSONL      = {TRAIN_JSONL}")
print(f"  TEST_JSONL       = {TEST_JSONL}")
print(f"  SMOKE_RUN_DIR    = {SMOKE_RUN_DIR}")
print(f"  STRICT_PARSE_GATE= {STRICT_PARSE_GATE:.0%}")
print(f"  FULL_MAX_STEPS   = {FULL_MAX_STEPS}")

Working directory: /users/PCS0229/imankhazrak/EHR-Agentic-AI
Config loaded.
  BASE_MODEL       = unsloth/gemma-4-e2b-it-unsloth-bnb-4bit
  TRAIN_JSONL      = /users/PCS0229/imankhazrak/EHR-Agentic-AI/dataset_for_unsloth/train.jsonl
  TEST_JSONL       = /users/PCS0229/imankhazrak/EHR-Agentic-AI/dataset_for_unsloth/test.jsonl
  SMOKE_RUN_DIR    = /users/PCS0229/imankhazrak/EHR-Agentic-AI/outputs/unsloth_mt_lora_response_only_smoke_20260520_185354
  STRICT_PARSE_GATE= 80%
  FULL_MAX_STEPS   = 1500


---
## Section 1 — Environment Check

Verifies Python, repo, git branch, CUDA, and installed package versions.
All cells in this section are safe to run on the login/OnDemand node.
`nvidia-smi` and `unsloth` will fail gracefully — this is expected.

In [15]:
print("=" * 60)
print("ENVIRONMENT CHECK")
print("=" * 60)

# Working directory and repo
print(f"\nCWD          : {Path.cwd()}")
print(f"PROJECT_DIR  : {PROJECT_DIR}")
print(f"Repo exists  : {PROJECT_DIR.is_dir()}")

# Python executable
print(f"\nPython       : {sys.executable}")
print(f"Version      : {sys.version.split()[0]}")

# Git branch
branch = run("git branch --show-current 2>&1")
print(f"\nGit branch   : {branch or '(unknown)'}")

# nvidia-smi (expected to fail on login node)
print("\nnvidia-smi:")
smi = run("nvidia-smi -L 2>&1")
if "failed" in smi.lower() or "error" in smi.lower() or not smi:
    print("  [expected] nvidia-smi unavailable on login/OnDemand node — GPU is only visible on compute nodes.")
else:
    print(smi)

# Package versions via importlib.metadata (avoids triggering unsloth CUDA init)
print("\nPackage versions:")
for pkg in ["torch", "transformers", "trl", "scikit-learn", "pandas", "numpy"]:
    try:
        ver = importlib.metadata.version(pkg)
        print(f"  {pkg:<20} {ver}")
    except importlib.metadata.PackageNotFoundError:
        print(f"  {pkg:<20} NOT FOUND")

# Unsloth — expected to fail on login node (no CUDA driver)
print("\nunsloth import:")
try:
    import unsloth  # noqa: F401
    print("  unsloth imported successfully.")
except Exception as exc:
    print(f"  [expected on login node] unsloth import skipped: {type(exc).__name__}: {exc}")

print("\n" + "=" * 60)

ENVIRONMENT CHECK

CWD          : /users/PCS0229/imankhazrak/EHR-Agentic-AI
PROJECT_DIR  : /users/PCS0229/imankhazrak/EHR-Agentic-AI
Repo exists  : True

Python       : /users/PCS0229/imankhazrak/venvs/unsloth_env/bin/python
Version      : 3.9.21

Git branch   : multitask

nvidia-smi:
GPU 0: Tesla V100-SXM2-32GB (UUID: GPU-d9810209-41d1-4aee-9d10-0a02b84f1783)

Package versions:
  torch                2.8.0
  transformers         NOT FOUND
  trl                  NOT FOUND
  scikit-learn         1.6.1
  pandas               2.3.3
  numpy                2.0.2

unsloth import:
  [expected on login node] unsloth import skipped: ModuleNotFoundError: No module named 'unsloth'



---
## Section 2 — Dataset Validation

Validates `dataset_for_unsloth/train.jsonl` and `dataset_for_unsloth/test.jsonl`.

Checks:
- Row counts
- All outputs parse as valid 7-task multitask JSON (`parse_multitask_output`)
- No row has the broken single-task schema (`{"prediction": ..., "probability": ...}`)
- Output field keys match expected task keys

In [16]:
import sys
sys.path.insert(0, str(PROJECT_DIR))

from src.llm.output_parser import MULTITASK_JSON_TASK_KEYS, parse_multitask_output

print("=" * 60)
print("DATASET VALIDATION")
print("=" * 60)
print(f"Expected task keys: {list(MULTITASK_JSON_TASK_KEYS)}\n")

for split, path in (("train", TRAIN_JSONL), ("test", TEST_JSONL)):
    print(f"--- {split} ({path}) ---")
    rows = load_jsonl(path)
    print(f"  Rows            : {len(rows)}")
    print(f"  Record keys     : {list(rows[0].keys())}")

    strict_ok    = 0
    wrong_single = 0
    missing_keys = 0

    for r in rows:
        out = r.get("output", "")
        if parse_multitask_output(out) is not None:
            strict_ok += 1
        try:
            obj = json.loads(out)
            if "prediction" in obj and "probability" in obj:
                wrong_single += 1
            if any(k not in obj for k in MULTITASK_JSON_TASK_KEYS):
                missing_keys += 1
        except Exception:
            missing_keys += 1

    print(f"  strict_ok       : {strict_ok} / {len(rows)}  {'OK' if strict_ok == len(rows) else 'FAIL'}")
    print(f"  wrong_single    : {wrong_single}  {'OK' if wrong_single == 0 else 'FAIL — single-task schema found!'}")
    print(f"  missing_keys    : {missing_keys}  {'OK' if missing_keys == 0 else 'FAIL'}")

    # Print first sample
    r0 = rows[0]
    print(f"\n  First record (pair_id={r0.get('pair_id')!r}):")
    instruction_preview = str(r0.get("instruction", ""))[:120].replace("\n", " ")
    input_preview       = str(r0.get("input", ""))[:200].replace("\n", " ")
    try:
        out_keys = list(json.loads(r0.get("output", "{}")).keys())
    except Exception:
        out_keys = ["(parse error)"]
    print(f"    instruction : {instruction_preview!r}")
    print(f"    input       : {input_preview!r}")
    print(f"    output keys : {out_keys}")
    print()

print("=" * 60)

DATASET VALIDATION
Expected task keys: ['lipid_next', 'diabetes_current', 'hypertension_current', 'obesity_current', 'cardio_next', 'kidney_next', 'stroke_next']

--- train (/users/PCS0229/imankhazrak/EHR-Agentic-AI/dataset_for_unsloth/train.jsonl) ---
  Rows            : 10031
  Record keys     : ['pair_id', 'instruction', 'input', 'output']
  strict_ok       : 10031 / 10031  OK
  wrong_single    : 0  OK
  missing_keys    : 0  OK

  First record (pair_id=1):
    instruction : 'You are given one current hospital visit narrative for a patient. Output exactly one JSON object with keys: lipid_next, '
    input       : '- Diagnoses made: Subendocardial infarction, initial episode of care; Cardiogenic shock; Blood in stool; Acute kidney failure, unspecified; Hypertensive chronic kidney disease, unspecified, with chron'
    output keys : ['lipid_next', 'diabetes_current', 'hypertension_current', 'obesity_current', 'cardio_next', 'kidney_next', 'stroke_next', 'reasoning']

--- test (/users/PC

---
## Section 3 — Tokenization and Label-Mask Audit Reports

The audit was already run and the reports are stored under `outputs/audits/`.

### What the audit confirms
- Processor type: `Gemma4Processor` (module `transformers.models.gemma4.processing_gemma4`)
- Inner tokenizer: `GemmaTokenizer` (accessed via `.tokenizer` attribute)
- EOS token: `<eos>` (ID = 1)
- Pad token: `<pad>` (ID = None in HF config)
- Supervision starts at the first `{` after `### Response:` — only the JSON response and EOS are supervised

Cell 3a displays existing reports. Cell 3b shows the commands to re-run audits if needed.

In [17]:
# Cell 3a — Display existing audit reports

AUDIT_DIR = PROJECT_DIR / "outputs/audits"

print("=" * 60)
print("AUDIT REPORTS")
print("=" * 60)

if AUDIT_DIR.is_dir():
    reports = sorted(AUDIT_DIR.glob("*.md"))
    print(f"Found {len(reports)} report(s) in {AUDIT_DIR}:\n")
    for r in reports:
        print(f"  {r.name}")
else:
    print(f"  [MISSING] Audit directory not found: {AUDIT_DIR}")
    reports = []

# Inline display of label-mask audit
LABEL_MASK_REPORT = AUDIT_DIR / "label_mask_audit_head5.md"
print("\n" + "=" * 60)
print(f"LABEL-MASK AUDIT REPORT: {LABEL_MASK_REPORT.name}")
print("=" * 60)
if LABEL_MASK_REPORT.is_file():
    lines = LABEL_MASK_REPORT.read_text().splitlines()
    print("\n".join(lines[:80]))
    if len(lines) > 80:
        print(f"\n... [{len(lines) - 80} more lines — read the full file at {LABEL_MASK_REPORT}]")
else:
    print("  [MISSING] Run the audit (see Cell 3b).")

# Smoke run label-mask audit (embedded in run dir)
SMOKE_AUDIT = SMOKE_RUN_DIR / "label_mask_audit.md"
print("\n" + "-" * 60)
print(f"SMOKE RUN LABEL-MASK AUDIT: {SMOKE_AUDIT}")
print("-" * 60)
if SMOKE_AUDIT.is_file():
    lines = SMOKE_AUDIT.read_text().splitlines()
    print("\n".join(lines[:40]))
    if len(lines) > 40:
        print(f"... [{len(lines) - 40} more lines]")
else:
    print("  [not found or smoke run not yet complete]")

AUDIT REPORTS
Found 7 report(s) in /users/PCS0229/imankhazrak/EHR-Agentic-AI/outputs/audits:

  label_mask_audit_head5.md
  label_mask_audit_strict_v2_head5.md
  unsloth_tokenization_audit_smoke.md
  unsloth_tokenization_audit_strict_v2_train_head5.md
  unsloth_tokenization_audit_test_head5.md
  unsloth_tokenization_audit_train_head5.md
  unsloth_tokenization_audit_train_longest5.md

LABEL-MASK AUDIT REPORT: label_mask_audit_head5.md
# Unsloth multitask label-mask audit

- samples: 5
- max_seq_length: 2048
- supervision starts at first `{` after `### Response:`

## Row 0 (pair_id=1)

| Metric | Value |
|--------|-------|
| tokenized_length | 772 |
| n_masked_tokens | 615 |
| n_supervised_tokens | 157 |
| pct_supervised | 20.34 |
| supervised_parse_ok | True |

### [A] Prompt/context (masked)

```text
### Instruction:
You are given one current hospital visit narrative for a patient. Output exactly one JSON object with keys: lipid_next, diabetes_current, hypertension_current, obesity_cur

In [18]:
# Cell 3b — Audit re-run commands (informational — not auto-executed)
#
# Verified CLI args (from --help):
#   audit_unsloth_multitask_tokenization.py : --num-samples  --output-md  (NOT --n-samples / --out-md)
#   train_unsloth_router.py                 : --debug-only   --debug-label-mask-md  --tokenizer-only
#
# To re-run the tokenization audit on CPU (tokenizer-only, no GPU needed):
#
#   conda run -n unsloth_env python scripts/audit_unsloth_multitask_tokenization.py \
#       --model-name unsloth/gemma-4-e2b-it-unsloth-bnb-4bit \
#       --jsonl dataset_for_unsloth/train.jsonl \
#       --tokenizer-only \
#       --sample-mode head \
#       --num-samples 5 \
#       --output-md outputs/audits/unsloth_tokenization_audit_rerun.md
#
# To re-run the label-mask audit (tokenizer-only + debug-only, CPU-safe — cannot start training):
#
#   conda run -n unsloth_env python scripts/train_unsloth_router.py \
#       --train-jsonl dataset_for_unsloth/train.jsonl \
#       --tokenizer-only \
#       --debug-only \
#       --debug-label-mask-check 5 \
#       --debug-label-mask-md outputs/audits/label_mask_audit_rerun.md

print("Audit re-run commands printed above as comments.")
print("Un-comment and call run() to execute:")
print()

TOKENIZATION_AUDIT_CMD = (
    f"conda run -n {CONDA_ENV} python scripts/audit_unsloth_multitask_tokenization.py "
    f"--model-name {BASE_MODEL} "
    f"--jsonl {TRAIN_JSONL} "
    "--tokenizer-only --sample-mode head --num-samples 5 "
    f"--output-md {AUDIT_DIR}/unsloth_tokenization_audit_rerun.md"
)

LABEL_MASK_AUDIT_CMD = (
    f"conda run -n {CONDA_ENV} python scripts/train_unsloth_router.py "
    f"--train-jsonl {TRAIN_JSONL} "
    "--tokenizer-only "
    "--debug-only "
    "--debug-label-mask-check 5 "
    f"--debug-label-mask-md {PROJECT_DIR}/outputs/audits/label_mask_audit_rerun.md"
)

print("Tokenization audit:")
print(f"  {TOKENIZATION_AUDIT_CMD}")
print()
print("Label-mask audit (--debug-only guarantees training cannot start):")
print(f"  {LABEL_MASK_AUDIT_CMD}")

# To run either audit, un-comment:
# print(run(TOKENIZATION_AUDIT_CMD))
# print(run(LABEL_MASK_AUDIT_CMD))

Audit re-run commands printed above as comments.
Un-comment and call run() to execute:

Tokenization audit:
  conda run -n unsloth_env python scripts/audit_unsloth_multitask_tokenization.py --model-name unsloth/gemma-4-e2b-it-unsloth-bnb-4bit --jsonl /users/PCS0229/imankhazrak/EHR-Agentic-AI/dataset_for_unsloth/train.jsonl --tokenizer-only --sample-mode head --num-samples 5 --output-md /users/PCS0229/imankhazrak/EHR-Agentic-AI/outputs/audits/unsloth_tokenization_audit_rerun.md

Label-mask audit (--debug-only guarantees training cannot start):
  conda run -n unsloth_env python scripts/train_unsloth_router.py --train-jsonl /users/PCS0229/imankhazrak/EHR-Agentic-AI/dataset_for_unsloth/train.jsonl --tokenizer-only --debug-only --debug-label-mask-check 5 --debug-label-mask-md /users/PCS0229/imankhazrak/EHR-Agentic-AI/outputs/audits/label_mask_audit_rerun.md


---
## Section 4 — Current SLURM Job Status

Checks the state of job `47576605` (smoke inference + strict scoring for the 500-step model).
Also shows the full user queue.

In [19]:
print("=" * 60)
print("SLURM JOB STATUS")
print("=" * 60)

# Full user queue
print("\n--- squeue (all user jobs) ---")
print(run("squeue -u $USER 2>&1") or "  (no jobs in queue)")

# Specific job detail via sacct
print(f"\n--- sacct for job {SMOKE_TEST_JOB_ID} ---")
sacct_out = run(
    f"sacct -j {SMOKE_TEST_JOB_ID} "
    "--format=JobID,JobName,State,ExitCode,Start,End,Elapsed,NodeList 2>&1"
)
print(sacct_out or "  (no sacct data)")

# Log file
LOG_FILE = PROJECT_DIR / f"logs/slurm-unsloth-response-only-smoke-test-{SMOKE_TEST_JOB_ID}.out"
ERR_FILE = PROJECT_DIR / f"logs/slurm-unsloth-response-only-smoke-test-{SMOKE_TEST_JOB_ID}.err"

print(f"\n--- Log file: {LOG_FILE} ---")
if LOG_FILE.is_file():
    print(tail_file(LOG_FILE, n=80))
else:
    print("  [not found — job may still be pending or log not yet written]")

print(f"\n--- Error file: {ERR_FILE} ---")
if ERR_FILE.is_file():
    print(tail_file(ERR_FILE, n=30))
else:
    print("  [not found]")

# Early look at strict summary if scoring already finished
SMOKE_STRICT_SUMMARY = SMOKE_RUN_DIR / "test_predictions_smoke50.strict_summary.json"
print(f"\n--- Strict summary JSON: {SMOKE_STRICT_SUMMARY.name} ---")
if SMOKE_STRICT_SUMMARY.is_file():
    summary = load_json(SMOKE_STRICT_SUMMARY)
    print(json.dumps(summary, indent=2)[:2000])
else:
    print("  [not found — scoring job not yet complete]")

print("\n" + "=" * 60)

SLURM JOB STATUS

--- squeue (all user jobs) ---
JOBID PARTITION     NAME     USER ST       TIME  NODES NODELIST(REASON)
          47578159   gpu-exp unsloth_ imankhaz  R      39:13      1 p0312
          47577383 gpubackfi ondemand imankhaz  R    1:35:40      1 p0351

--- sacct for job 47576605 ---
JobID           JobName      State ExitCode               Start                 End    Elapsed        NodeList 
------------ ---------- ---------- -------- ------------------- ------------------- ---------- --------------- 
47576605     unsloth_r+  COMPLETED      0:0 2026-05-20T20:38:21 2026-05-20T21:39:45   01:01:24           p0342 
47576605.ba+      batch  COMPLETED      0:0 2026-05-20T20:38:21 2026-05-20T21:39:45   01:01:24           p0342 
47576605.ex+     extern  COMPLETED      0:0 2026-05-20T20:38:21 2026-05-20T21:39:45   01:01:24           p0342

--- Log file: /users/PCS0229/imankhazrak/EHR-Agentic-AI/logs/slurm-unsloth-response-only-smoke-test-47576605.out ---
=== Unsloth response-o

---
## Section 5 — Load Smoke Results

Loads results from the 500-step response-only smoke run:
```
outputs/unsloth_mt_lora_response_only_smoke_20260520_185354/
```

**Known context:**
- The 100-step smoke run (same directory root) had **0/50 strict parse coverage** and produced the wrong
  single-task schema `{"prediction": "No", "probability": "0.00", "reasoning": "..."}` — this was a training
  issue (full-sequence loss instead of response-only loss).
- The 500-step run was trained with **response-only loss masking** and its inference + scoring
  job `47576605` is pending. Results will appear once that job completes.

Update `SMOKE_RUN_DIR` in Section 0 if you run a new smoke train.

In [ ]:
print("=" * 60)
print("SMOKE RUN RESULTS")
print(f"RUN_DIR: {SMOKE_RUN_DIR}")
print("=" * 60)

# --- Artifact existence checks ---
print("\nArtifact checks:")
final_lora_ok    = check_file(SMOKE_RUN_DIR / "final_lora",                              "final_lora/")
manifest_ok      = check_file(SMOKE_RUN_DIR / "train_manifest.json",                     "train_manifest.json")
preds_ok         = check_file(SMOKE_RUN_DIR / "test_predictions_smoke50.jsonl",          "test_predictions_smoke50.jsonl")
summary_ok       = check_file(SMOKE_RUN_DIR / "test_predictions_smoke50.strict_summary.json",
                               "test_predictions_smoke50.strict_summary.json")
metrics_ok       = check_file(SMOKE_RUN_DIR / "test_predictions_smoke50_metrics_strict.md",
                               "test_predictions_smoke50_metrics_strict.md")

# --- Train manifest ---
if manifest_ok:
    print("\nTrain manifest:")
    manifest = load_json(SMOKE_RUN_DIR / "train_manifest.json")
    for k in ["run_name", "model_name", "train_max_samples", "eval_max_samples",
               "max_steps", "save_steps", "loss_masking", "supervision_starts_at"]:
        print(f"  {k:<30} {manifest.get(k, '(missing)')}")

# --- Predictions JSONL ---
smoke_coverage_pct = None
smoke_preds = []

if preds_ok:
    smoke_preds = load_jsonl(SMOKE_RUN_DIR / "test_predictions_smoke50.jsonl")
    n_total   = len(smoke_preds)
    n_ok      = sum(1 for r in smoke_preds if r.get("pred_parse_ok"))
    smoke_coverage_pct = n_ok / n_total if n_total > 0 else 0.0
    print(f"\nPredictions JSONL:")
    print(f"  Total rows         : {n_total}")
    print(f"  pred_parse_ok      : {n_ok}")
    print(f"  Strict coverage    : {smoke_coverage_pct:.1%}")
    show_predictions(smoke_preds, n=5, label="First 5 predictions")
    show_failed_predictions(smoke_preds, n=5)
else:
    print("\n  Predictions JSONL not found — submit Section 9 to run inference.")

# --- Strict summary JSON ---
if summary_ok:
    summary = load_json(SMOKE_RUN_DIR / "test_predictions_smoke50.strict_summary.json")
    cov = summary.get("coverage", {})
    n_rows  = cov.get("n_rows", 0)
    n_usable = cov.get("rows_with_usable_prediction", 0)
    if n_rows > 0:
        smoke_coverage_pct = n_usable / n_rows
    print(f"\nStrict summary (from scoring script):")
    print(f"  parse_mode                 : {summary.get('parse_mode')}")
    print(f"  n_rows                     : {n_rows}")
    print(f"  rows_with_usable_prediction: {n_usable}")
    print(f"  strict_parse_coverage      : {smoke_coverage_pct:.1%}")
    macro = summary.get("macro", {})
    if macro:
        print(f"  macro accuracy             : {macro.get('accuracy')}")
        print(f"  macro f1                   : {macro.get('f1')}")
else:
    print("\n  Strict summary JSON not found — scoring job not yet complete.")

# --- Metrics markdown ---
METRICS_MD = SMOKE_RUN_DIR / "test_predictions_smoke50_metrics_strict.md"
if metrics_ok:
    print("\nMetrics markdown (first 60 lines):")
    lines = METRICS_MD.read_text().splitlines()
    print("\n".join(lines[:60]))

print("\n" + "=" * 60)

---
## Section 6 — Strict Parse Coverage Gate

The model is ready for full training only if it can generate valid 7-task JSON reliably.

Gate threshold: **`STRICT_PARSE_GATE = 80%`**

If the gate fails, see Section 7 for the strict-v2 dataset approach and Section 8 to submit a new smoke run.

In [ ]:
print("=" * 60)
print("STRICT PARSE COVERAGE GATE")
print("=" * 60)

GATE_PASSED = False

if smoke_coverage_pct is None:
    print("\n  Coverage unknown — predictions JSONL or strict summary not yet available.")
    print("  Run inference + scoring (Section 9) and reload this section.")
else:
    gate_pct = STRICT_PARSE_GATE
    GATE_PASSED = smoke_coverage_pct >= gate_pct

    print(f"\n  Strict parse coverage  : {smoke_coverage_pct:.1%}")
    print(f"  Gate threshold         : {gate_pct:.0%}")
    print(f"  Gate passed            : {GATE_PASSED}")

    if GATE_PASSED:
        print("\n  RECOMMENDATION: Coverage is acceptable.")
        print("  Proceed to full training (Section 10).")
        print("  Submit: sbatch slurm/train_unsloth_mt_natural_dist.slurm")
    else:
        print("\n  RECOMMENDATION: Coverage is below threshold.")
        print("  DO NOT submit full training yet.")
        print("  Options:")
        print("    1. Wait for job 47576605 to complete and reload (coverage may improve).")
        print("    2. Build strict-v2 dataset (Section 7) and rerun smoke training (Section 8).")
        print("    3. Review failed predictions above to understand the failure mode.")

print("\n" + "=" * 60)
print(f"GATE_PASSED = {GATE_PASSED}  (used by Section 10)")

---
## Section 7 — [CONDITIONAL] Strict-v2 Dataset Branch

> **Only needed if the gate in Section 6 fails.**

The 100-step smoke run demonstrated a failure mode where the model generates the wrong
single-task schema `{"prediction": "No", "probability": "0.00", "reasoning": "..."}` instead
of the required 7-task JSON. If this persists after 500 steps of response-only training,
the dataset labels themselves should be reinforced.

### Strict-v2 plan

1. **Stronger instruction text** — make the required key names and structure explicit, e.g.:
   > *"Output exactly one JSON object with the following top-level keys: `lipid_next`,
   > `diabetes_current`, `hypertension_current`, `obesity_current`, `cardio_next`,
   > `kidney_next`, `stroke_next`. Each key maps to `{"prediction": "Yes"|"No", "probability": float}`."*

2. **Compact JSON output** — use `separators=(",", ":")` when writing gold outputs to reduce token count and
   make the JSON boundary crisper for the masking heuristic.

3. **Shorter reasoning field** — keep `reasoning` under ~15 tokens so the supervised region
   is dominated by the structured JSON prediction, not free-text.

4. **New output folder** — write to `dataset_for_unsloth_multitask_strict_v2/` so the
   original dataset is preserved.

### Commands

```bash
# 1. Export strict-v2 dataset (update export script with compact JSON + short reasoning)
conda run -n unsloth_env python src/scripts/export_multitask_unsloth_jsonl.py \
    --out-dir dataset_for_unsloth_multitask_strict_v2

# 2. Re-run tokenization audit on v2
conda run -n unsloth_env python scripts/audit_unsloth_multitask_tokenization.py \
    --model-name unsloth/gemma-4-e2b-it-unsloth-bnb-4bit \
    --jsonl dataset_for_unsloth_multitask_strict_v2/train.jsonl \
    --tokenizer-only --sample-mode head --num-samples 5 \
    --output-md outputs/audits/unsloth_tokenization_audit_strict_v2_train_head5.md

# 3. Re-run label-mask audit on v2 (--debug-only prevents training from starting)
conda run -n unsloth_env python scripts/train_unsloth_router.py \
    --train-jsonl dataset_for_unsloth_multitask_strict_v2/train.jsonl \
    --tokenizer-only \
    --debug-only \
    --debug-label-mask-check 5 \
    --debug-label-mask-md outputs/audits/label_mask_audit_strict_v2.md

# 4. Submit new smoke training pointing at v2 dataset
#    (update SLURM script or pass env vars, or create a new SLURM script)
```

After building strict-v2, update `TRAIN_JSONL`, `TEST_JSONL`, and `SMOKE_RUN_DIR`
in Section 0 and re-run Sections 2–6.

---
## Section 8 — [SLURM] Submit Smoke Training

> **Warning: Only run this cell if no acceptable smoke model exists,
> or if you intentionally want a new smoke run (e.g. after building strict-v2 dataset).**
>
> The 500-step response-only smoke model already exists at:
> `outputs/unsloth_mt_lora_response_only_smoke_20260520_185354/`
>
> To submit a new smoke run, un-comment the `run(sbatch_cmd)` line below.

In [ ]:
sbatch_smoke_train = (
    f"cd {PROJECT_DIR} && "
    "sbatch slurm/train_unsloth_response_only_smoke.slurm"
)

print("=" * 60)
print("[SLURM] SMOKE TRAINING SUBMISSION")
print("=" * 60)
print(f"\nSLURM script   : slurm/train_unsloth_response_only_smoke.slurm")
print(f"Partition      : {PARTITION}")
print(f"Account        : {ACCOUNT}")
print(f"Max steps      : {SMOKE_MAX_STEPS}")
print(f"Train samples  : {SMOKE_TRAIN_SAMPLES}")
print(f"Eval samples   : {SMOKE_EVAL_SAMPLES}")
print(f"\nCommand:\n  {sbatch_smoke_train}")
print("\n*** Un-comment the line below to submit ***")

# out = run(sbatch_smoke_train)
# print(out)
# print("After job finishes, update SMOKE_RUN_DIR in Section 0 to the new output directory.")
# print("Then re-run Sections 4-6.")

---
## Section 9 — [SLURM] Submit Smoke Inference + Scoring

Runs inference on 50 test rows using the smoke LoRA, then runs strict scoring.

Two modes:
- **Full** (inference + scoring): submit if `test_predictions_smoke50.jsonl` does not exist
- **Score-only**: submit if predictions already exist but strict metrics markdown is missing

> Job `47576605` was already submitted for `SMOKE_RUN_DIR`. Monitor it in Section 4.
> Un-comment below only if you need to re-submit or submit for a different run directory.

In [ ]:
print("=" * 60)
print("[SLURM] SMOKE INFERENCE + SCORING SUBMISSION")
print("=" * 60)

PREDS_JSONL = SMOKE_RUN_DIR / "test_predictions_smoke50.jsonl"
preds_already_exist = PREDS_JSONL.is_file()

print(f"\nSMOKE_RUN_DIR       : {SMOKE_RUN_DIR}")
print(f"MODEL_LORA          : {SMOKE_RUN_DIR / 'final_lora'}")
print(f"Predictions exist   : {preds_already_exist}")

# Full inference + scoring
sbatch_smoke_test_full = (
    f"cd {PROJECT_DIR} && "
    f"RUN_DIR={SMOKE_RUN_DIR} "
    "sbatch slurm/test_unsloth_response_only_smoke.slurm"
)

# Score-only (predictions already exist)
sbatch_smoke_score_only = (
    f"cd {PROJECT_DIR} && "
    f"SCORE_ONLY=1 RUN_DIR={SMOKE_RUN_DIR} "
    "sbatch slurm/test_unsloth_response_only_smoke.slurm"
)

print(f"\nFull inference + scoring command:")
print(f"  {sbatch_smoke_test_full}")
print(f"\nScore-only command (use if predictions JSONL already exists):")
print(f"  {sbatch_smoke_score_only}")
print()

if preds_already_exist:
    print("Predictions JSONL already exists.")
    print("Recommend: use SCORE_ONLY mode (un-comment below).")
else:
    print("Predictions JSONL not found.")
    print("Recommend: use full inference + scoring (un-comment below).")

print("\n*** Un-comment ONE of the lines below to submit ***")

# Full inference + scoring:
# out = run(sbatch_smoke_test_full)
# print(out)

# Score-only (predictions exist):
# out = run(sbatch_smoke_score_only)
# print(out)

---
## Section 10 — [SLURM] Full Training

> **Run only after the smoke gate passes in Section 6 (`GATE_PASSED = True`).**

### SLURM script: `slurm/train_unsloth_mt_natural_dist.slurm`

Key parameters:
| Parameter | Value |
|-----------|-------|
| max-steps | 1500 |
| save-steps | 50 |
| gradient-accumulation-steps | 8 |
| per-device-train-batch-size | 1 |
| wall time | 24h |
| GPU | V100-32G |
| loss masking | response-only (default in `train_unsloth_router.py`) |

> **Note:** `train_unsloth_router.py` defaults to `--train-on-response-only` (the flag is `default=True`).
> `train_unsloth_mt_natural_dist.slurm` does not pass `--train-on-full-text`, so response-only masking
> is active.

> **Before submitting: run the inspection cell below (Cell 10a) to print the full SLURM script.**
> Verify it does **not** contain `--train-on-full-text`. Only submit after confirming this.

In [ ]:
# Cell 10a — Inspect full-training SLURM script before submitting
#
# Read and print slurm/train_unsloth_mt_natural_dist.slurm in full.
# Confirm:
#   - does NOT contain --train-on-full-text  (response-only is the default)
#   - uses the correct account, partition, and wall time

FULL_TRAIN_SLURM = PROJECT_DIR / "slurm/train_unsloth_mt_natural_dist.slurm"

print("=" * 60)
print("FULL-TRAINING SLURM SCRIPT INSPECTION")
print(f"Path: {FULL_TRAIN_SLURM}")
print("=" * 60)

if not FULL_TRAIN_SLURM.is_file():
    print(f"  [MISSING] {FULL_TRAIN_SLURM}")
    print("  Cannot submit full training without this script.")
else:
    content = FULL_TRAIN_SLURM.read_text()
    print(content)
    print("\n" + "-" * 60)
    print("SAFETY CHECKS:")

    has_full_text_flag = "--train-on-full-text" in content
    has_response_only  = "--train-on-response-only" in content or "response_only" in content
    has_account        = "#SBATCH --account=" in content
    has_partition      = "#SBATCH --partition=" in content

    print(f"  --train-on-full-text present : {has_full_text_flag}  {'DANGER — do not submit!' if has_full_text_flag else 'OK'}")
    print(f"  response-only referenced     : {has_response_only}  (default=True in train script; OK if False here)")
    print(f"  #SBATCH --account present    : {has_account}")
    print(f"  #SBATCH --partition present  : {has_partition}")

    if has_full_text_flag:
        print("\n  WARNING: script contains --train-on-full-text.")
        print("  This disables response-only masking. Do NOT submit until this is removed.")
    else:
        print("\n  Script does not pass --train-on-full-text.")
        print("  Response-only masking is active (default=True in train_unsloth_router.py).")
        print("  Safe to submit if GATE_PASSED = True.")

In [ ]:
print("=" * 60)
print("[SLURM] FULL TRAINING SUBMISSION")
print("=" * 60)

if not GATE_PASSED:
    print("\n  GATE NOT PASSED — full training is not recommended.")
    print("  Strict parse coverage must reach {:.0%} before submitting.".format(STRICT_PARSE_GATE))
    print("  See Section 6 for details.")
else:
    print("\n  Gate passed — full training may be submitted.")

sbatch_full_train = (
    f"cd {PROJECT_DIR} && "
    "sbatch slurm/train_unsloth_mt_natural_dist.slurm"
)

print(f"\nSLURM script   : slurm/train_unsloth_mt_natural_dist.slurm")
print(f"Max steps      : {FULL_MAX_STEPS}")
print(f"Save steps     : {FULL_SAVE_STEPS}")
print(f"Wall time      : 24h")
print(f"Loss masking   : response-only (default)")
print(f"\nCommand:\n  {sbatch_full_train}")
print("\n*** Un-comment the line below to submit (only if GATE_PASSED is True) ***")

# out = run(sbatch_full_train)
# print(out)
# print("Monitor progress with: squeue -u $USER")
# print("Tail the log with: tail -f logs/slurm-unsloth_mt_natural-<JOBID>.out")

---
## Section 11 — Experiment Summary

Scans all `outputs/unsloth_mt_lora_*/` directories and builds a summary DataFrame
with training parameters, artifact existence, and strict parse coverage.

In [ ]:
print("=" * 60)
print("EXPERIMENT SUMMARY")
print("=" * 60)

OUTPUTS_DIR = PROJECT_DIR / "outputs"
run_dirs = sorted(OUTPUTS_DIR.glob("unsloth_mt_lora_*"), key=lambda p: p.name)
print(f"\nFound {len(run_dirs)} run director{'y' if len(run_dirs) == 1 else 'ies'} under {OUTPUTS_DIR}\n")

rows = []
for rd in run_dirs:
    if not rd.is_dir():
        continue

    # Load manifest
    mf_path = rd / "train_manifest.json"
    if mf_path.is_file():
        mf = load_json(mf_path)
    else:
        mf = {}

    # Artifact existence
    final_lora_exists = (rd / "final_lora").exists()
    preds_path        = rd / "test_predictions_smoke50.jsonl"
    preds_exists      = preds_path.is_file()
    summary_path      = rd / "test_predictions_smoke50.strict_summary.json"
    summary_exists    = summary_path.is_file()
    metrics_path      = rd / "test_predictions_smoke50_metrics_strict.md"
    metrics_exists    = metrics_path.is_file()

    # Strict parse coverage
    strict_coverage = None
    if summary_exists:
        try:
            s = load_json(summary_path)
            cov = s.get("coverage", {})
            n   = cov.get("n_rows", 0)
            ok  = cov.get("rows_with_usable_prediction", 0)
            strict_coverage = ok / n if n > 0 else 0.0
        except Exception:
            strict_coverage = None
    elif preds_exists:
        try:
            preds = load_jsonl(preds_path)
            n  = len(preds)
            ok = sum(1 for r in preds if r.get("pred_parse_ok"))
            strict_coverage = ok / n if n > 0 else 0.0
        except Exception:
            strict_coverage = None

    # Recommendation
    if strict_coverage is None:
        rec = "no results yet"
    elif strict_coverage >= STRICT_PARSE_GATE:
        rec = "GATE PASSED — proceed to full training"
    else:
        rec = f"gate failed ({strict_coverage:.0%} < {STRICT_PARSE_GATE:.0%})"

    rows.append({
        "run_dir"             : rd.name,
        "max_steps"           : mf.get("max_steps", ""),
        "train_samples"       : mf.get("train_max_samples", ""),
        "eval_samples"        : mf.get("eval_max_samples", ""),
        "final_lora_exists"   : final_lora_exists,
        "predictions_exists"  : preds_exists,
        "strict_parse_coverage": (
            f"{strict_coverage:.1%}" if strict_coverage is not None else "—"
        ),
        "metrics_md_exists"   : metrics_exists,
        "recommendation"      : rec,
    })

df = pd.DataFrame(rows)

if df.empty:
    print("  No run directories found.")
else:
    pd.set_option("display.max_colwidth", None)
    pd.set_option("display.max_rows", 50)
    print(df.to_string(index=False))

print("\n" + "=" * 60)
print(f"Generated: {datetime.utcnow().strftime('%Y-%m-%d %H:%M UTC')}")